# Data Integration Using SQL

This notebook uses SQLite to explore and integrate the cleaned 2023–2025 Maryland education datasets. The analysis sample is restricted to grade-level Math and ELA MCAP assessments for grades 3–8, which are integrated with school-level attendance and student demographic/context variables.

The cleaned datasets were created in `01_data_preparation.ipynb`.

In [1]:
from pathlib import Path
import pandas as pd
import sqlite3

PROJECT_ROOT = Path.home() / "Documents" / "Maryland Education Project"
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DATA_DIR = DATA_DIR / "processed"

## Load Processed Data

The cleaned Math, ELA, Attendance, and Special Services datasets from 2023–2025 are loaded from the `processed` folder.

School and LEA identifiers are loaded as strings rather than numeric values because they are categorical identifiers and may contain leading zeros.

In [2]:
# keep school identifiers as strings so leading zeros are preserved
id_dtypes = {
    "lea_id": "string",
    "school_id": "string"
}

math = pd.read_csv(
    PROCESSED_DATA_DIR / "mcap_math_2023_2025.csv",
    dtype=id_dtypes
)

ela = pd.read_csv(
    PROCESSED_DATA_DIR / "mcap_ela_2023_2025.csv",
    dtype=id_dtypes
)

attendance = pd.read_csv(
    PROCESSED_DATA_DIR / "attendance_2023_2025.csv",
    dtype=id_dtypes
)

special = pd.read_csv(
    PROCESSED_DATA_DIR / "special_services_2023_2025.csv",
    dtype=id_dtypes
)

## Create SQLite Database
The four processed datasets are loaded into a local SQLite database so they can be explored and integrated using SQL.

In [3]:
# create location where SQLite database file will live
db_path = PROCESSED_DATA_DIR / "maryland_education.db"

# open connection between Python and database
conn = sqlite3.connect(db_path)

In [4]:
# write each processed DataFrame as a table in SQLite database
math.to_sql("mcap_math", conn, if_exists="replace", index=False)
ela.to_sql("mcap_ela", conn, if_exists="replace", index=False)
attendance.to_sql("attendance", conn, if_exists="replace", index=False)
special.to_sql("special_services", conn, if_exists="replace", index=False)

5179

### Verify Database Tables

Before using the database, confirm that all four processed datasets were successfully written as SQLite tables.

In [5]:
query = """
SELECT name
FROM sqlite_master
WHERE type = 'table';
"""

pd.read_sql(query, conn)

,name
0,mcap_math
1,mcap_ela
2,attendance
3,special_services


### Verify Table Row Counts

Compare the number of records written to each SQLite table with the processed DataFrames to confirm that no rows were lost when the data were transferred into the database.

In [6]:
query = """
SELECT 'mcap_math' AS table_name, COUNT(*) AS row_count
FROM mcap_math

-- combines rows from multiple query results vertically
UNION ALL

SELECT 'mcap_ela', COUNT(*)
FROM mcap_ela

UNION ALL

SELECT 'attendance', COUNT(*)
FROM attendance

UNION ALL

SELECT 'special_services', COUNT(*)
FROM special_services;
"""

pd.read_sql(query, conn)

,table_name,row_count
0,mcap_math,17906
1,mcap_ela,15797
2,attendance,5123
3,special_services,5179


## Examine MCAP Assessment Structure

Before defining the analysis sample, examine the available Math and ELA assessments across 2023–2025. The primary analysis will focus on grade-level assessments for grades 3–8 so that proficiency patterns can be compared across subjects and grades.

In [7]:
query = """
SELECT
    assessment,
    COUNT(*) AS row_count
FROM mcap_math
GROUP BY assessment
ORDER BY assessment;
"""

pd.read_sql(query, conn)

,assessment,row_count
0,Algebra 1,1543
1,Algebra 2,486
2,Geometry,813
3,Mathematics 03,2620
4,Mathematics 04,2623
5,Mathematics 05,2611
6,Mathematics 06,1098
7,Mathematics 07,1052
8,Mathematics 08,1043
9,Mathematics All Grades,4017


In [8]:
query = """
SELECT
    assessment,
    COUNT(*) AS row_count
FROM mcap_ela
GROUP BY assessment
ORDER BY assessment;
"""

pd.read_sql(query, conn)

,assessment,row_count
0,English Language Arts 03,2620
1,English Language Arts 04,2622
2,English Language Arts 05,2610
3,English Language Arts 06,1100
4,English Language Arts 07,1051
5,English Language Arts 08,1052
6,English Language Arts 10,724
7,English Language Arts All Grades,4018


### Examine School-Type Structure

Attendance and Special Services contain records for different school-type populations. Check which school-type categories are available in each dataset.

In [9]:
pd.read_sql("""
SELECT
    school_type,
    COUNT(*) AS row_count
FROM special_services
GROUP BY school_type
ORDER BY school_type;
""", conn)

,school_type,row_count
0,All,493
1,Elementary,2798
2,High,777
3,Middle,1111


In [10]:
pd.read_sql("""
SELECT
    school_type,
    COUNT(*) AS row_count
FROM attendance
GROUP BY school_type
ORDER BY school_type;
""", conn)

,school_type,row_count
0,All,471
1,Elementary,2762
2,High,777
3,Middle,1113


### Validate School-Type Record Uniqueness

Before using school type in later joins, verify that each combination of year, school, and school type appears only once in Attendance and Special Services.

In [11]:
pd.read_sql("""
SELECT
    'attendance' AS table_name,
    COUNT(*) AS duplicate_rows
FROM (
    SELECT year, lea_id, school_id, school_type
    FROM attendance
    GROUP BY year, lea_id, school_id, school_type
    HAVING COUNT(*) > 1
)

UNION ALL

SELECT
    'special_services',
    COUNT(*)
FROM (
    SELECT year, lea_id, school_id, school_type
    FROM special_services
    GROUP BY year, lea_id, school_id, school_type
    HAVING COUNT(*) > 1
);
""", conn)

,table_name,duplicate_rows
0,attendance,0
1,special_services,0


In [12]:
query = """
SELECT
    m.assessment,
    a.school_type,
    COUNT(*) AS row_count
FROM mcap_math AS m
-- match Math assessment records to attendance records by year and school
INNER JOIN attendance AS a
    ON m.year = a.year
    AND m.lea_id = a.lea_id
    AND m.school_id = a.school_id
WHERE m.assessment IN ('Algebra 1', 'Algebra 2', 'Geometry')
GROUP BY m.assessment, a.school_type
ORDER BY m.assessment, a.school_type;
"""

pd.read_sql(query, conn)

,assessment,school_type,row_count
0,Algebra 1,All,268
1,Algebra 1,Elementary,212
2,Algebra 1,High,701
3,Algebra 1,Middle,897
4,Algebra 2,All,29
5,Algebra 2,Elementary,8
6,Algebra 2,High,339
7,Algebra 2,Middle,172
8,Geometry,All,75
9,Geometry,Elementary,50


The results indicate that some schools have Attendance records for multiple school types. Joining MCAP and Attendance using only year and school identifiers could therefore produce multiple matches for a single assessment record.

### Examine Multiple School Types

Before integrating MCAP and school-level context, check whether a school can have multiple school-type records within the same year. If so, joining the datasets using only the year and school identifiers could produce multiple matches for a single MCAP record.

In [13]:
query = """
SELECT
    year,
    lea_id,
    school_id,
    COUNT(*) AS school_type_count
FROM attendance
-- find schools with multiple school-type records within the same year
GROUP BY year, lea_id, school_id
HAVING COUNT(*) > 1
ORDER BY school_type_count DESC;
"""

pd.read_sql(query, conn)

,year,lea_id,school_id,school_type_count
0,2023,02,1274,4
1,2023,02,3414,4
2,2023,02,4054,4
3,2023,02,4074,4
4,2023,02,4304,4
...,...,...,...,...
466,2024,02,3232,2
467,2024,03,0062,2
468,2024,30,0086,2
469,2025,02,3232,2


There are 471 school-year combinations with more than one Attendance `school_type` record. Therefore, year and school identifiers alone are not sufficient to uniquely match MCAP records to school-level context for these schools. School type will also be considered when integrating the datasets.


## Define the Analysis Sample

The primary analysis focuses on grade-level Math and ELA MCAP performance for grades 3–8. Restricting the sample to assessments explicitly corresponding to grades 3-8 allows proficiency patterns to be compared across subjects, grades, and years.

Course-based Math assessments (Algebra 1, Geometry, and Algebra 2) and All Grades results are excluded from the primary analysis because they do not correspond directly to a single grade-level assessment.

In [14]:
# keep ELA assessments explicitly tied to grades 3-8
query = """
SELECT 
    *,
    -- extract the two-digit grade from the end of the assessment name
    CAST(SUBSTR(assessment, -2, 2) AS INT) AS grade
FROM mcap_ela
WHERE assessment IN (
    'English Language Arts 03',
    'English Language Arts 04',
    'English Language Arts 05',
    'English Language Arts 06',
    'English Language Arts 07',
    'English Language Arts 08'
);
"""

ela_grades_3_8 = pd.read_sql(query, conn)

ela_grades_3_8.shape

(11055, 19)

In [15]:
ela_grades_3_8[["assessment", "grade"]].drop_duplicates()

,assessment,grade
0,English Language Arts 03,3
1,English Language Arts 04,4
2,English Language Arts 05,5
9,English Language Arts 06,6
10,English Language Arts 07,7
11,English Language Arts 08,8


In [16]:
# keep Math assessments explicitly tied to grades 3–8
query = """
SELECT 
    *,
    CAST(SUBSTR(assessment, -2, 2) AS INT) AS grade
FROM mcap_math
WHERE assessment IN (
    'Mathematics 03',
    'Mathematics 04',
    'Mathematics 05',
    'Mathematics 06',
    'Mathematics 07',
    'Mathematics 08'
);
"""

math_grades_3_8 = pd.read_sql(query, conn)

math_grades_3_8.shape

(11047, 19)

In [17]:
math_grades_3_8[["assessment", "grade"]].drop_duplicates()

,assessment,grade
0,Mathematics 03,3
1,Mathematics 04,4
2,Mathematics 05,5
9,Mathematics 06,6
10,Mathematics 07,7
11,Mathematics 08,8


### Validate Analysis Sample Coverage
Compare the number of Math and ELA records by grade and year to check whether the grade-level analysis samples have similar coverage.

In [18]:
query = """
SELECT assessment, year, COUNT(*) AS count
FROM mcap_ela
WHERE assessment IN (
    'English Language Arts 03',
    'English Language Arts 04',
    'English Language Arts 05',
    'English Language Arts 06',
    'English Language Arts 07',
    'English Language Arts 08'
)
GROUP BY assessment, year
"""

ela_counts = pd.read_sql(query, conn)

In [19]:
ela_counts

,assessment,year,count
0,English Language Arts 03,2023,876
1,English Language Arts 03,2024,873
2,English Language Arts 03,2025,871
3,English Language Arts 04,2023,873
4,English Language Arts 04,2024,875
5,English Language Arts 04,2025,874
6,English Language Arts 05,2023,871
7,English Language Arts 05,2024,869
8,English Language Arts 05,2025,870
9,English Language Arts 06,2023,386


In [20]:
query = """
SELECT assessment, year, COUNT(*) AS count
FROM mcap_math
WHERE assessment IN (
    'Mathematics 03',
    'Mathematics 04',
    'Mathematics 05',
    'Mathematics 06',
    'Mathematics 07',
    'Mathematics 08'
)
GROUP BY assessment, year
"""

math_counts = pd.read_sql(query, conn)

In [21]:
math_counts

,assessment,year,count
0,Mathematics 03,2023,876
1,Mathematics 03,2024,873
2,Mathematics 03,2025,871
3,Mathematics 04,2023,873
4,Mathematics 04,2024,876
5,Mathematics 04,2025,874
6,Mathematics 05,2023,871
7,Mathematics 05,2024,870
8,Mathematics 05,2025,870
9,Mathematics 06,2023,386


Math and ELA have very similar record counts across grades and years, indicating comparable coverage in the two grade-level analysis samples. Small differences occur for some grade-year combinations, particularly in grade 8.

## Combine Math and ELA Analysis Samples

The grade-level Math and ELA records are combined into a single long-format dataset, with each row representing a school-year-grade-subject record. This structure allows subject, grade, and year to be analyzed within the same dataset.

In [22]:
query = """
-- keep only grade-level math assessments for grades 3-8
SELECT
    *,
    CAST(SUBSTR(assessment, -2, 2) AS INT) AS grade
FROM mcap_math
WHERE assessment IN (
    'Mathematics 03',
    'Mathematics 04',
    'Mathematics 05',
    'Mathematics 06',
    'Mathematics 07',
    'Mathematics 08'
)

UNION ALL -- stack ELA records below Math records without removing rows

-- keep only grade-level ELA assessments for grades 3-8
SELECT
    *,
    CAST(SUBSTR(assessment, -2, 2) AS INT) AS grade
FROM mcap_ela
WHERE assessment IN (
    'English Language Arts 03',
    'English Language Arts 04',
    'English Language Arts 05',
    'English Language Arts 06',
    'English Language Arts 07',
    'English Language Arts 08'
);
"""

mcap_grades_3_8 = pd.read_sql(query, conn)

mcap_grades_3_8.shape

(22102, 19)

## Integrate School-Level Context

Attendance and Special Services contain separate records by school type for some schools. Grade-level MCAP records are first matched to the expected school type: Elementary for grades 3-5 and Middle for grades 6-8.

Some schools do not follow these typical grade configurations. If the expected school type is not available, but the school has only one Attendance or Special Services record for that year, that record is used instead. If multiple records are available and the expected school type cannot be matched, the contextual variables are left missing rather than choosing a record arbitrarily.

In [23]:
query = """
-- combine grade-level Math and ELA records into one analysis sample and create grade column
WITH combined_mcap AS (
    SELECT
        *,
        -- extract grade number from end of assessment name
        CAST(SUBSTR(assessment, -2, 2) AS INT) AS grade
    FROM mcap_math
    WHERE assessment IN (
        'Mathematics 03',
        'Mathematics 04',
        'Mathematics 05',
        'Mathematics 06',
        'Mathematics 07',
        'Mathematics 08'
    )

    -- stacks grade 3-8 Math and ELA records
    UNION ALL

    SELECT
        *,
        CAST(SUBSTR(assessment, -2, 2) AS INT) AS grade
    FROM mcap_ela
    WHERE assessment IN (
        'English Language Arts 03',
        'English Language Arts 04',
        'English Language Arts 05',
        'English Language Arts 06',
        'English Language Arts 07',
        'English Language Arts 08'
    )
),

-- assign the school type normally expected for each assessment grade
mcap_with_school_type AS (
    SELECT
        *,
        CASE
            WHEN grade BETWEEN 3 AND 5 THEN 'Elementary'
            WHEN grade BETWEEN 6 AND 8 THEN 'Middle'
        END AS school_type
    FROM combined_mcap
),

-- count Attendance records available for each school-year
-- used later to identify schools with only one possible attendance match
attendance_counts AS (
    SELECT
        year,
        lea_id,
        school_id,
        COUNT(*) AS attendance_record_count
    FROM attendance
    GROUP BY year, lea_id, school_id
),

-- count Special Services records for the same purpose
special_counts AS (
    SELECT
        year,
        lea_id,
        school_id,
        COUNT(*) AS special_record_count
    FROM special_services
    GROUP BY year, lea_id, school_id
)

SELECT
    m.*,
    a.school_type AS attendance_school_type,
    a.attendance_pct,
    a.more_20_pct,
    a.chronic_absentee_pct,
    s.school_type AS special_school_type,
    s.farms_pct,
    s.multilingual_learner_pct,
    s.students_with_disabilities_pct,
    s.economically_disadvantaged_pct,
    s.gifted_talented_pct,
    s.total_student_count
FROM mcap_with_school_type AS m

-- add the number of Attendance records available for each school-year
LEFT JOIN attendance_counts AS ac
    ON m.year = ac.year
    AND m.lea_id = ac.lea_id
    AND m.school_id = ac.school_id

-- match attendance by expected school type
-- if only one Attendance record exists, use it instead
LEFT JOIN attendance AS a
    ON m.year = a.year
    AND m.lea_id = a.lea_id
    AND m.school_id = a.school_id
    AND (
        m.school_type = a.school_type
        OR ac.attendance_record_count = 1
    )

-- add number of Special Services records available for each school-year
LEFT JOIN special_counts AS sc
    ON m.year = sc.year
    AND m.lea_id = sc.lea_id
    AND m.school_id = sc.school_id

-- apply same matching rules to Special Services
LEFT JOIN special_services AS s
    ON m.year = s.year
    AND m.lea_id = s.lea_id
    AND m.school_id = s.school_id
    AND (
        m.school_type = s.school_type
        OR sc.special_record_count = 1
    );
"""

mcap_integrated = pd.read_sql(query, conn)
mcap_integrated.shape

(22102, 31)

### Validate Integrated Dataset
Check whether the joins preserved the original MCAP analysis sample without creating duplicate records, and identify any records that could not be matched to Attendance or Special Services.

In [24]:
# check missingness in matched Special Services variables
mcap_integrated[
    [
        "farms_pct",
        "multilingual_learner_pct",
        "students_with_disabilities_pct",
        "economically_disadvantaged_pct",
        "gifted_talented_pct",
        "total_student_count"
    ]
].isna().sum()

farms_pct                         59
multilingual_learner_pct          59
students_with_disabilities_pct    59
economically_disadvantaged_pct    59
gifted_talented_pct               59
total_student_count               59
dtype: int64

In [25]:
# verify one record per school-year-grade-subject combination
duplicates = mcap_integrated.duplicated(
    subset=["year", "lea_id", "school_id", "grade", "subject"]
).sum()

duplicates

np.int64(0)

In [26]:
# check missingness in matched Attendance variables
mcap_integrated[
    ["attendance_school_type", "attendance_pct", "more_20_pct", "chronic_absentee_pct"]
].isna().sum()

attendance_school_type    73
attendance_pct            73
more_20_pct               73
chronic_absentee_pct      73
dtype: int64

In [27]:
# confirm that integration did not change the number of MCAP records
print("Original MCAP records:", len(mcap_grades_3_8))
print("Integrated records:", len(mcap_integrated))

Original MCAP records: 22102
Integrated records: 22102


The final integrated dataset retains all 22,102 MCAP records with no duplicate school-year-grade-subject records. There are 73 records (0.33%) without matched Attendance data and 59 records (0.27%) without matched Special Services data. These records are retained with missing contextual values.

### Save Analysis-Ready Dataset
Save the final integrated dataset for use in subsequent exploratory analysis and modeling.

In [28]:
output_path = PROCESSED_DATA_DIR / "analysis_ready_2023_2025.csv"
mcap_integrated.to_csv(output_path, index=False)